# Tufts Dental Database: Mahalanobis screening of tooth crops

This notebook produces the Tufts refusal rates and distance distributions (Table 8, Fig. 5). DENTEX test tooth crops and Tufts tooth crops are embedded with the same frozen encoders (Dental ViT-L and BiomedCLIP). The Mahalanobis distance is fitted on the DENTEX training embeddings of the seed-42 split, and the refusal threshold is the 95th percentile of the DENTEX test distances.

Tufts tooth boxes come from `Segmentation/teeth_bbox.json`, which is index-aligned with `Expert/expert.json`. The boxes are stored on a portrait annotation canvas, whereas the radiographs are upright landscape images. Section 6 maps the boxes with an extent-calibrated counter-clockwise transform, and Section 6a draws the mapped boxes on one radiograph so that the mapping can be checked before anything is embedded.

Run `03_three_splits_and_delta_sweep.ipynb` first: this notebook loads the DENTEX embeddings from its cache.

## 1. Environment and device

In [ ]:
import sys, subprocess
def pipq(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p], check=False)
# Packages are listed in requirements.txt. Uncomment to install them into the active kernel.
# pipq("torch","torchvision","scikit-learn","pillow","matplotlib","tqdm",
#      "huggingface_hub","open_clip_torch","timm","ml_collections")
import torch, numpy as np, os, glob, json, math
DEVICE="mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## 2. Paths: DENTEX (embedding cache) and Tufts (radiographs, tooth boxes, expert labels)

In [ ]:
DENTEX_DIR = "/path/to/DENTEX/training_data/quadrant-enumeration-disease"
TUFTS_DIR  = "/path/to/Tufts"
TUFTS_IMG_DIR     = os.path.join(TUFTS_DIR, "Radiographs")          # folder with the panoramic .jpg files
TUFTS_BBOX_JSON   = os.path.join(TUFTS_DIR, "Segmentation", "teeth_bbox.json")
TUFTS_EXPERT_JSON = os.path.join(TUFTS_DIR, "Expert", "expert.json")

for p in [DENTEX_DIR, TUFTS_IMG_DIR]:
    assert os.path.isdir(p), f"folder not found: {p}"
for p in [TUFTS_BBOX_JSON, TUFTS_EXPERT_JSON]:
    assert os.path.exists(p), f"file not found: {p}"

CACHE_DIR  = os.path.join(DENTEX_DIR, "_results_option1", "emb_cache")
OUTPUT_DIR = os.path.join(DENTEX_DIR, "_results_option1")
assert os.path.isdir(CACHE_DIR), ("DENTEX embedding cache not found. Run 03_three_splits_and_delta_sweep.ipynb first.")
print("DENTEX cache:", CACHE_DIR)
print("cached backbones:", [os.path.basename(p) for p in glob.glob(os.path.join(CACHE_DIR,"emb_*.npy"))])

## 3. Rebuild DENTEX tooth indexing and the seed-42 grouped split

The Mahalanobis reference is the DENTEX training embeddings under the same grouped split (seed 42) as `03_three_splits_and_delta_sweep.ipynb`.

In [ ]:
ANN_PATH=None
for root,_,files in os.walk(DENTEX_DIR):
    for f in files:
        if f.endswith(".json"):
            try: j=json.load(open(os.path.join(root,f)))
            except Exception: continue
            if isinstance(j,dict) and "annotations" in j and any("category_id_3" in a for a in j.get("annotations",[])):
                ANN_PATH=os.path.join(root,f); break
    if ANN_PATH: break
coco=json.load(open(ANN_PATH))
cats3=sorted(coco["categories_3"],key=lambda c:c["id"]); NUM_DX=len(cats3)
id2idx={c["id"]:i for i,c in enumerate(cats3)}
img_file={im["id"]:im["file_name"] for im in coco["images"]}
tooth={}
for a in coco["annotations"]:
    key=(a["image_id"],a.get("category_id_1",-1),a.get("category_id_2",-1))
    if key not in tooth: tooth[key]={"image_id":a["image_id"]}
teeth=list(tooth.values())
IMG_OF=np.array([t["image_id"] for t in teeth])

def grouped_split(img_of,seed,vf=0.15,tf=0.15):
    ids=sorted(set(img_of.tolist())); rng=np.random.RandomState(seed); rng.shuffle(ids)
    n=len(ids); nte=int(n*tf); nva=int(n*vf)
    te=set(ids[:nte]); va=set(ids[nte:nte+nva]); tr_,va_,te_=[],[],[]
    for i,im in enumerate(img_of):
        (te_ if im in te else va_ if im in va else tr_).append(i)
    return np.array(tr_),np.array(va_),np.array(te_)
tr_idx,va_idx,te_idx=grouped_split(IMG_OF,42)
print(f"DENTEX teeth {len(teeth)} | train {len(tr_idx)} test {len(te_idx)}")

## 4. Load the same frozen backbones (BiomedCLIP and dental ViT-L)

In [ ]:
import open_clip
from tqdm.auto import tqdm
def load_biomedclip():
    model,_,preprocess=open_clip.create_model_and_transforms("hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
    return model.visual.to(DEVICE).eval(), preprocess

DENTAL_OK=False; dental_model=None; dental_preprocess=None
try:
    if not os.path.isdir("ViT-pytorch"):
        subprocess.run(["git","clone","--depth","1","https://github.com/jeonsworld/ViT-pytorch"],check=True)
    sys.path.insert(0,"ViT-pytorch")
    from models.modeling import VisionTransformer, CONFIGS
    from huggingface_hub import hf_hub_download
    ck=hf_hub_download(repo_id="Bryceee/Teeth_Visual_Experts_Models",
        filename="Teeth_Visual_Experts_ViT_L_periapical_images_cls_7diseases.bin")
    state=torch.load(ck,map_location="cpu")
    if isinstance(state,dict) and "state_dict" in state: state=state["state_dict"]
    pos=state["transformer.embeddings.position_embeddings"].shape[1]
    DENTAL_IMG=int(round(math.sqrt(pos-1)))*16; nh=state["head.weight"].shape[0]
    cfg=CONFIGS["ViT-L_16"]; m=VisionTransformer(cfg,img_size=DENTAL_IMG,num_classes=nh,zero_head=False)
    m.load_state_dict(state,strict=False); m=m.to(DEVICE).eval()
    import torchvision.transforms as T
    dental_preprocess=T.Compose([T.Resize((DENTAL_IMG,DENTAL_IMG)),T.ToTensor(),T.Normalize([0.5]*3,[0.5]*3)])
    class DE(torch.nn.Module):
        def __init__(s,v): super().__init__(); s.v=v
        @torch.no_grad()
        def forward(s,x): seq,_=s.v.transformer(x); return seq[:,0]
    dental_model=DE(m).to(DEVICE).eval(); DENTAL_OK=True
    print("dental ViT-L loaded (input",DENTAL_IMG,")")
except Exception as e:
    print("dental ViT-L not loaded:",repr(e),"Continuing with BiomedCLIP only.")

## 5. Load cached DENTEX embeddings (no re-extraction)

In [ ]:
DENTEX_EMB={}
for p in glob.glob(os.path.join(CACHE_DIR,"emb_*.npy")):
    name=os.path.basename(p)[4:-4]   # emb_<name>.npy
    DENTEX_EMB[name]=np.load(p)
    print(f"{name}: {DENTEX_EMB[name].shape}")

## 6. Tufts tooth boxes and the extent-calibrated counter-clockwise mapping

`teeth_bbox.json` and `expert.json` are index-aligned lists. Each box is `[x1, y1, x2, y2]` on the portrait annotation canvas. Coordinates are normalised by the observed box extents over all boxes, then rotated counter-clockwise onto the upright image: `nx = (ya - AY_MIN)/(AY_MAX - AY_MIN)`, `ny = (xa - AX_MIN)/(AX_MAX - AX_MIN)`, `image_x = nx*W`, `image_y = ny*H`. Image-level label: `Description == "Within normal limits"` is normal, anything else abnormal.

In [ ]:
from PIL import Image
bbox_items = json.load(open(TUFTS_BBOX_JSON))
exp_items  = json.load(open(TUFTS_EXPERT_JSON))
assert len(bbox_items) == len(exp_items), "teeth_bbox.json and expert.json must be index-aligned"

def item_boxes(item):
    out = []
    for o in (item.get("Label") or {}).get("objects", []) or []:
        b = o.get("bounding box")
        if isinstance(b, (list, tuple)) and len(b) == 4:
            out.append([float(v) for v in b])
    return out

ALL_BOXES = [b for it in bbox_items for b in item_boxes(it)]
AX_MIN = min(min(b[0], b[2]) for b in ALL_BOXES); AX_MAX = max(max(b[0], b[2]) for b in ALL_BOXES)
AY_MIN = min(min(b[1], b[3]) for b in ALL_BOXES); AY_MAX = max(max(b[1], b[3]) for b in ALL_BOXES)
print(f"boxes: {len(ALL_BOXES)} | canvas extents x[{AX_MIN:.0f},{AX_MAX:.0f}] y[{AY_MIN:.0f},{AY_MAX:.0f}]")

def map_box(b, W, H):
    # extent-calibrated CCW map: portrait annotation canvas -> upright landscape image
    xs, ys = [], []
    for xa, ya in ((b[0], b[1]), (b[2], b[3])):
        nx = (ya - AY_MIN) / (AY_MAX - AY_MIN)
        ny = (xa - AX_MIN) / (AX_MAX - AX_MIN)
        xs.append(nx * W); ys.append(ny * H)
    return [min(xs), min(ys), max(xs), max(ys)]

def crop_box(img, box, pad=0.15, min_size=8):
    W, H = img.size; x0, y0, x1, y1 = box
    w, h = x1 - x0, y1 - y0
    if w < min_size or h < min_size: return None
    cx0, cy0 = max(0, int(x0 - w*pad)), max(0, int(y0 - h*pad))
    cx1, cy1 = min(W, int(x1 + w*pad)), min(H, int(y1 + h*pad))
    if cx1 - cx0 < min_size or cy1 - cy0 < min_size: return None
    return img.crop((cx0, cy0, cx1, cy1))

def tufts_image_path(ext_id):
    stem = os.path.splitext(ext_id)[0]
    for ext in (".jpg", ".JPG", ".jpeg", ".png"):
        p = os.path.join(TUFTS_IMG_DIR, stem + ext)
        if os.path.exists(p): return p
    return None

TUFTS_ITEMS = []   # (image_path, [canvas boxes], abnormal 0/1)
for it_b, it_e in zip(bbox_items, exp_items):
    p = tufts_image_path(it_e.get("External ID", ""))
    bxs = item_boxes(it_b)
    if p is None or not bxs: continue
    abn = 0 if it_e.get("Description", "").strip().lower() == "within normal limits" else 1
    TUFTS_ITEMS.append((p, bxs, abn))
print("Tufts images with boxes:", len(TUFTS_ITEMS))

### 6a. Visual check of the mapped boxes

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
p, bxs, _ = TUFTS_ITEMS[0]
im = Image.open(p).convert("RGB"); W, H = im.size
fig, ax = plt.subplots(figsize=(14, 7)); ax.imshow(im)
for b in bxs:
    x0, y0, x1, y1 = map_box(b, W, H)
    ax.add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False, edgecolor="lime", lw=1.2))
ax.set_title(os.path.basename(p) + ": mapped tooth boxes")
plt.tight_layout(); plt.show()

### 6b. Extract Tufts tooth crops and embed them with the same backbones (cached)

In [ ]:
TUFTS_CROPS, TUFTS_OWNER_IDX, TUFTS_ABN = [], [], []
skipped = 0
for gi, (p, bxs, abn) in enumerate(tqdm(TUFTS_ITEMS, desc="tufts crops")):
    im = Image.open(p).convert("RGB"); W, H = im.size
    for b in bxs:
        c = crop_box(im, map_box(b, W, H))
        if c is None: skipped += 1; continue
        TUFTS_CROPS.append(c); TUFTS_OWNER_IDX.append(gi); TUFTS_ABN.append(abn)
TUFTS_OWNER_IDX = np.array(TUFTS_OWNER_IDX); TUFTS_ABN = np.array(TUFTS_ABN)
print(f"Tufts tooth crops: {len(TUFTS_CROPS)} | skipped {skipped}")

@torch.no_grad()
def embed_crops(model, preprocess, crops, bs=32, l2norm=True):
    feats = []
    for i in tqdm(range(0, len(crops), bs), desc="embed", leave=False):
        x = torch.stack([preprocess(c) for c in crops[i:i+bs]]).to(DEVICE)
        f = model(x)
        if isinstance(f, (tuple, list)): f = f[0]
        f = f.float().cpu()
        if l2norm: f = torch.nn.functional.normalize(f, dim=1)
        feats.append(f.numpy())
    return np.concatenate(feats)

TUFTS_EMB = {}
def cache_tufts(name, model, pre):
    fpath = os.path.join(CACHE_DIR, f"tufts_toothcrop_emb_{name}.npy")
    if os.path.exists(fpath):
        arr = np.load(fpath)
        if len(arr) == len(TUFTS_CROPS):
            print(f"{name}: loaded cache {arr.shape}"); return arr
    arr = embed_crops(model, pre, TUFTS_CROPS); np.save(fpath, arr)
    print(f"{name}: {arr.shape}"); return arr

bmc_model, bmc_pre = load_biomedclip()
TUFTS_EMB["biomedclip"] = cache_tufts("biomedclip", bmc_model, bmc_pre)
if DENTAL_OK:
    TUFTS_EMB["dental_vitl"] = cache_tufts("dental_vitl", dental_model, dental_preprocess)
print("tufts backbones:", list(TUFTS_EMB.keys()))

## 7. Mahalanobis distance: DENTEX test vs Tufts, and the refusal rates

Fitted on DENTEX train. Threshold at the 95th percentile of DENTEX test, then the fraction of Tufts crops above that threshold is reported.

In [ ]:
def maha_fit(Xtr):
    mu=Xtr.mean(0); cov=np.cov(Xtr.T)+np.eye(Xtr.shape[1])*1e-2; ci=np.linalg.inv(cov)
    return lambda X: np.einsum("ni,ij,nj->n",X-mu,ci,X-mu)

backbones=[n for n in DENTEX_EMB if n in TUFTS_EMB]
fig,axes=plt.subplots(1,len(backbones),figsize=(6*len(backbones),4),squeeze=False)
ood={}
for j,name in enumerate(backbones):
    scorer=maha_fit(DENTEX_EMB[name][tr_idx])
    d_in =scorer(DENTEX_EMB[name][te_idx])     # DENTEX test (in-distribution)
    d_ood=scorer(TUFTS_EMB[name])              # Tufts tooth crops (OOD)
    thr=np.quantile(d_in,0.95)
    r_in=float((d_in>thr).mean()); r_ood=float((d_ood>thr).mean())
    ood[name]={"n_dentex_test":int(len(d_in)),"n_tufts":int(len(d_ood)),
               "dentex_test_mean":float(d_in.mean()),"tufts_mean":float(d_ood.mean()),
               "threshold_p95":float(thr),"refuse_dentex":r_in,"refuse_tufts":r_ood}
    ax=axes[0][j]
    lo,hi=np.percentile(np.concatenate([d_in,d_ood]),[1,99])
    bins=np.linspace(lo,hi,40)
    ax.hist(d_in,bins=bins,alpha=0.6,label="DENTEX test",color="#1a3a5c",density=True)
    ax.hist(d_ood,bins=bins,alpha=0.6,label="Tufts (OOD)",color="#c9a227",density=True)
    ax.axvline(thr,ls="--",color="red",label="refuse threshold")
    ax.set_title(f"{name}\nTufts refused {r_ood:.0%} vs DENTEX {r_in:.0%}")
    ax.set_xlabel("Mahalanobis distance"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,"tufts_ood_distance.png"),dpi=140); plt.show()
for n,v in ood.items():
    print(f"{n}: DENTEX-test refused {v['refuse_dentex']:.1%} (n={v['n_dentex_test']}) | "
          f"Tufts refused {v['refuse_tufts']:.1%} (n={v['n_tufts']})")

## 7a. AUROC, AUPR and FPR95 of the Mahalanobis score (Table 7)

In [ ]:
# AUROC, AUPR and FPR95 of the Mahalanobis score: DENTEX test teeth versus Tufts tooth crops.
# Same scorer and arrays as Section 7. FPR95 is the fraction of DENTEX test teeth flagged when 95%
# of the Tufts crops are flagged.
import numpy as np
def ood_metrics(d_in, d_ood):
    s=np.concatenate([d_ood,d_in]); y=np.concatenate([np.ones(len(d_ood)),np.zeros(len(d_in))])
    order=np.argsort(s); r=np.empty_like(order,float); r[order]=np.arange(1,len(s)+1)
    npos,nneg=int(y.sum()),int((y==0).sum())
    auroc=(r[y==1].sum()-npos*(npos+1)/2)/(npos*nneg)
    thr=np.unique(s)[::-1]; P,R=[],[]
    for t in thr:
        tp=((s>=t)&(y==1)).sum(); fp=((s>=t)&(y==0)).sum(); fn=((s<t)&(y==1)).sum()
        P.append(tp/(tp+fp) if tp+fp else 1.0); R.append(tp/(tp+fn) if tp+fn else 0.0)
    P,R=np.array(P),np.array(R); o=np.argsort(R); aupr=np.trapz(P[o],R[o])
    fpr95=float((d_in>=np.quantile(d_ood,0.05)).mean())   # FPR at 95% OOD recall
    return auroc,aupr,fpr95

print(f"{'backbone':14s} {'AUROC':>7s} {'AUPR':>7s} {'FPR95':>7s}")
ood_std={}
for name in backbones:
    scorer=maha_fit(DENTEX_EMB[name][tr_idx])
    d_in =scorer(DENTEX_EMB[name][te_idx])
    d_ood=scorer(TUFTS_EMB[name])
    a,p,f=ood_metrics(d_in,d_ood)
    ood_std[name]={"auroc":round(a,4),"aupr":round(p,4),"fpr95":round(f,4)}
    print(f"{name:14s} {a:7.3f} {p:7.3f} {f:7.3f}")

## 8. Normal vs abnormal Tufts radiographs (per-image mean distance)

In [ ]:
for name in backbones:
    scorer=maha_fit(DENTEX_EMB[name][tr_idx]); d=scorer(TUFTS_EMB[name])
    per={}
    for dist,gi in zip(d,TUFTS_OWNER_IDX): per.setdefault(int(gi),[]).append(dist)
    norm=[np.mean(v) for gi,v in per.items() if TUFTS_ITEMS[gi][2]==0]
    abn =[np.mean(v) for gi,v in per.items() if TUFTS_ITEMS[gi][2]==1]
    print(f"{name}: Tufts-normal mean dist={np.mean(norm):.1f} (n={len(norm)}) | "
          f"Tufts-abnormal mean dist={np.mean(abn):.1f} (n={len(abn)})")

## 9. Save results

In [ ]:
import datetime
out={"timestamp":datetime.datetime.now().isoformat(timespec="seconds"),
     "device":DEVICE,"backbones":backbones,
     "tufts_input":{"type":"tooth crops","source":"Segmentation/teeth_bbox.json",
                    "mapping":"extent-calibrated CCW","n_crops":len(TUFTS_CROPS),"n_skipped":skipped,
                    "pad":0.15,"min_size":8},
     "ood":ood}
p=os.path.join(OUTPUT_DIR,"option1_tufts_ood_toothcrops.json")
json.dump(out,open(p,"w"),indent=2)
print("Saved ->",p)